# Final Visualization Report: Stealthy Poisoning in RAG

**Project question.** This project studies whether poisoned documents in a RAG knowledge base can be detected and mitigated before they degrade question-answering behavior.

**Visualization goal.** The original hypothesis was that poisoned documents would move into a distinct region of embedding space. The visual evidence says the opposite: poisoned and clean documents strongly overlap. That overlap is the main finding, not a dashboard bug.

**Conceptual pivot.** Frame this as the **Stealthy Poisoning Paradox**: the attacks change factual truth enough to hurt RAG, but preserve topic and writing style enough to remain embedded with clean documents.

## Executive Summary

- Evaluated **5 poisoned KB variants**: factual swaps at 10%, 20%, and 30%; semantic distortions at 10% and 30%.
- Detection metrics table contains **25 detector rows** across Isolation Forest, LOF, neural classifier, GPT-2 perplexity, and LLM-judge rows where available.
- Available detector AUCs range from **0.4514** to **0.6001** with mean **0.5270**, which is close to random chance.
- Best observed detector F1 is **0.4663** for `neural_classifier` on `semantic_0.3`, but this must be read with precision/recall because broad flagging inflates recall.
- Latest clean RAG baseline uses **1000 examples** with EM **0.0610** and token-F1 **0.1069**.
- Final RAG rows compare **undefended**, **naive threshold**, and **top-K capped** strategies across **5 variant settings**.
- Naive threshold mitigation ranges from token-F1 **0.0023** to **0.0423** and still documents the `factual_0.2` empty-KB failure.
- Top-K capped mitigation covers all five variants with mean token-F1 **0.0690**, matching the available undefended baseline **0.0690** rather than improving it.
- Mitigation filter recall ranges from **0.730** to **1.000**, while precision ranges from **0.118** to **0.303**.
- Mitigation trade-off: naive threshold defense can be destructive, while the safer capped defense preserves the KB but is performance-neutral.
- LLM-judge AUC is intentionally unavailable because current judge outputs are binary yes/no verdicts, not continuous ROC scores.

## How To Read This Report

Orange points represent true poisoned documents. Blue points represent clean documents inside the poisoned KB. Gray points are clean reference documents. Good separation would mean orange points forming their own cluster or region. Instead, they overlap with blue and gray points, indicating that the embedding representation is largely invariant to these poisoning edits.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, SVG, Image, Markdown

ROOT = Path.cwd()
while not (ROOT / 'results').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

rag = pd.read_csv(ROOT / 'results' / 'tables' / 'module4_table1_rag_accuracy.csv')
det = pd.read_csv(ROOT / 'results' / 'tables' / 'module4_table2_detector_performance.csv')
mit = pd.read_csv(ROOT / 'results' / 'tables' / 'module4_table3_mitigation_filter_summary.csv')


## Results Tables

The tables below are the report source of truth. Table 1 records the final RAG evaluation rows, Table 2 records detector-level precision/recall/F1/AUC, and Table 3 compares threshold versus top-K mitigation behavior.


### Table 1: RAG Accuracy

This table includes the clean baseline, the available poisoned undefended row, the old threshold-mitigated rows, and the new top-K capped rows for all five poisoning variants.


In [ ]:
display(rag)


### Table 2: Detector Performance

This table is the main quantitative evidence for the detection layer. AUC values near 0.5 mean the detector score ranking is close to random; F1 values must be interpreted with `n_flagged` because some detectors get recall by flagging many documents.


In [ ]:
display(det)


### Table 3: Mitigation Filter Summary

This table makes the mitigation trade-off explicit. The threshold strategy shows the old high-risk filter behavior; the top-K capped rows show the safer strategy that keeps the KB usable.


In [ ]:
display(mit)


## Embedding Space Visualizations

These are the most important project figures. The expected success pattern would be orange poisoned points separating from clean points. The actual pattern is heavy overlap, which explains why embedding-space detectors and score-based detectors struggle.


### Embedding Projection: factual_0.1

Gray points are clean reference documents, blue points are clean documents in the poisoned KB, and orange points are true poisoned documents. The lack of separation supports the finding that embedding-space anomaly detection is weak for these attacks.


In [ ]:
fig = "results/figures/embedding_space_factual_0.1.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Embedding Projection: factual_0.2

Gray points are clean reference documents, blue points are clean documents in the poisoned KB, and orange points are true poisoned documents. The lack of separation supports the finding that embedding-space anomaly detection is weak for these attacks.


In [ ]:
fig = "results/figures/embedding_space_factual_0.2.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Embedding Projection: factual_0.3

Gray points are clean reference documents, blue points are clean documents in the poisoned KB, and orange points are true poisoned documents. The lack of separation supports the finding that embedding-space anomaly detection is weak for these attacks.


In [ ]:
fig = "results/figures/embedding_space_factual_0.3.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Embedding Projection: semantic_0.1

Gray points are clean reference documents, blue points are clean documents in the poisoned KB, and orange points are true poisoned documents. The lack of separation supports the finding that embedding-space anomaly detection is weak for these attacks.


In [ ]:
fig = "results/figures/embedding_space_semantic_0.1.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Embedding Projection: semantic_0.3

Gray points are clean reference documents, blue points are clean documents in the poisoned KB, and orange points are true poisoned documents. The lack of separation supports the finding that embedding-space anomaly detection is weak for these attacks.


In [ ]:
fig = "results/figures/embedding_space_semantic_0.3.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Embedding Space Overview

Small-multiple view of all five poisoning variants. Orange poisoned documents remain embedded inside the same local neighborhoods as clean documents, which is the central visual result.


In [ ]:
fig = "results/figures/embedding_space_all_variants_preview.png"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Report Asset: Embedding Small Multiples

Presentation PNG showing all five variants. The local projection is t-SNE because UMAP is not installed, but the visual finding is the required overlap pattern: poisoned points stay inside clean neighborhoods.


In [ ]:
fig = "results/figures/umap_small_multiples.png"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


## RAG Accuracy Context

This figure compares undefended retrieval, naive threshold mitigation, and capped top-K mitigation. The final story is not that defense wins; it is that naive defense can break the KB, while safe top-K defense is stable but neutral.


### RAG Strategy Comparison

The final RAG comparison separates the old threshold filter from the safer top-K capped filter. Threshold mitigation can collapse the KB; top-K capped mitigation keeps the system usable but does not improve token-F1 over undefended retrieval.


In [ ]:
fig = "results/figures/rag_accuracy_vs_contamination_available.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Report Asset: RAG Strategy Comparison

Presentation PNG comparing undefended retrieval, naive threshold mitigation, and capped top-K mitigation. The safe top-K fix prevents KB collapse but remains neutral versus undefended F1.


In [ ]:
fig = "results/figures/rag_accuracy_curves.png"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


## Detector Performance Curves

These figures summarize whether the detectors can separate poisoned documents from clean documents. The main result is weak separation: most AUCs are close to chance.


### Detector F1 and AUC by Variant

Detector AUCs cluster near 0.5, so the detectors are close to random ranking in embedding/score space. F1 values should be interpreted together with flagging behavior because some methods over-flag.


In [ ]:
fig = "results/figures/detector_f1_auc_by_variant.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Precision, Recall, and F1 at Low/High Poisoning

This chart compares detector tradeoffs at 10% and 30% contamination. The neural model often gets high recall by flagging broadly, while unsupervised methods miss most poisoned documents under automatic thresholds.


In [ ]:
fig = "results/figures/detector_precision_recall_f1_10_30.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### Report Asset: Detector AUC Comparison

Presentation PNG showing detector AUCs clustered around the random-guess line. This supports the Stealthy Poisoning Paradox narrative.


In [ ]:
fig = "results/figures/detector_comparison_bar.png"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


## Mitigation Filter Behavior

Mitigation is evaluated here as a filtering step. The filter removes many poisoned examples, but it also removes clean documents, so end-to-end RAG recovery remains a separate required measurement.


### Mitigation Filter Precision and Recall

The filter can remove many poisoned documents, but precision is low because many clean documents are dropped too. This explains why defended RAG recovery must be evaluated separately before claiming end-to-end improvement.


In [ ]:
fig = "results/figures/mitigation_filter_precision_recall.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


## ROC Diagnostics

ROC curves give a threshold-independent view of detector quality. Curves near the diagonal reinforce the embedding-overlap finding.


### ROC Curves: factual_0.1

ROC curves near the diagonal indicate weak score separation between poisoned and clean documents. This is consistent with the embedding-projection overlap and near-random AUC values.


In [ ]:
fig = "results/figures/roc_curves_factual_0.1.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### ROC Curves: factual_0.2

ROC curves near the diagonal indicate weak score separation between poisoned and clean documents. This is consistent with the embedding-projection overlap and near-random AUC values.


In [ ]:
fig = "results/figures/roc_curves_factual_0.2.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### ROC Curves: factual_0.3

ROC curves near the diagonal indicate weak score separation between poisoned and clean documents. This is consistent with the embedding-projection overlap and near-random AUC values.


In [ ]:
fig = "results/figures/roc_curves_factual_0.3.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### ROC Curves: semantic_0.1

ROC curves near the diagonal indicate weak score separation between poisoned and clean documents. This is consistent with the embedding-projection overlap and near-random AUC values.


In [ ]:
fig = "results/figures/roc_curves_semantic_0.1.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


### ROC Curves: semantic_0.3

ROC curves near the diagonal indicate weak score separation between poisoned and clean documents. This is consistent with the embedding-projection overlap and near-random AUC values.


In [ ]:
fig = "results/figures/roc_curves_semantic_0.3.svg"
path = ROOT / fig
if path.suffix.lower() == '.svg':
    display(SVG(filename=str(path)))
elif path.suffix.lower() == '.png':
    display(Image(filename=str(path)))
else:
    display(Markdown(f'Unsupported figure type: `{fig}`'))


## Final Integration Notes

All required visualization inputs are now present for the final story: embedding projections, detector metrics, mitigation summaries, poisoned KB change logs, and RAG strategy rows. The threshold `factual_0.2` failure is retained as evidence of naive mitigation risk, while top-K capped `factual_0.2` shows the safer fix is stable.

Additional caveat: LLM-judge AUC is unavailable because the current judge produces binary yes/no verdicts rather than continuous scores for ROC ranking.


## Failure Analysis

The table below embeds the key before/after examples directly from the poisoned change logs. These true poisoned documents received low anomaly scores, so the detectors treated them as especially clean-like.

| category | variant | doc_id | example | why_it_fails |
| --- | --- | --- | --- | --- |
| Category 1: Hard Factual Swaps | factual_0.1 | wiki_00095_chunk_018 | Before: ...th Milan). In addition, Barcelona became the first football club to win six out of six competitions in a single year (2009), completing the...<br>After: ...th Milan). In addition, Barcelona became the first football club to win six out of six competitions in a single year (2010), completing the... | A single date/fact token changes while the surrounding topic, entities, and writing style remain stable, so the embedding barely shifts. |
| Category 2: Semantic Overlap | semantic_0.1 | wiki_00095_chunk_018 | Before: They have won fifteen titles and were runners-up three times. Real Madrid is also the most successful club in the Intercontinental Cup (three...<br>After: They have secured fifteen titles and have been runners-up three times. Real Madrid is recognized as a highly successful club in the Interconti... | The rewrite is fluent and topical. all-mpnet-base-v2 preserves topic neighborhood more strongly than factual truth, so the passage looks clean. |
| Category 3: Boundary Cases | factual_0.2 | wiki_00778_chunk_002 | Before: ...ames Wilson and John Blair Jr. as its associate justices. All six were confirmed by the U.S. Senate on September 26, 1789. Harrison decline...<br>After: ...ames Wilson and John Blair Jr. as its associate justices. All six were confirmed by the U.S. Senate on September 26, 1788. Harrison decline... | The score sits near the expected poison-rate cutoff, so small threshold changes can flip the decision and make mitigation unstable. |


## Final Takeaway

The final visualization story is coherent: poisoned documents do not separate in mpnet embedding space, detector AUCs stay close to random chance, naive thresholding can damage or empty the KB, and capped top-K mitigation prevents collapse without improving answer quality. This supports the Stealthy Poisoning Paradox: the attack changes truth, but preserves enough topic and style to evade embedding-space defenses.
